In [1]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"

import numpy as np
import timeit
import csv
import numba
from numba import set_num_threads
from sklearn.datasets import make_regression

import sys

In [2]:
current_dir = os.getcwd()
# current_dir

In [3]:
sys.path.append(os.path.abspath(os.path.join(os.path.dirname(current_dir), '../')))

In [4]:
# os.path.dirname(current_dir)

In [5]:
# sys.path

In [7]:
# -----------------------------
# IMPORT YOUR FUNCTIONS
# -----------------------------
from Experiments.Numba.OMP_Implementations.OMP_numba_parallel import (
    OMP_serial,
    OMP_numba_serial,
    OMP_numba_jit,
    OMP_numba_njit,
    OMP_numba_njit_parallel,
    OMP_numba_njit_parallel_pbatches,
)

In [8]:
import importlib
OMP_A_V1_1 = importlib.import_module('Implementations.AK-SVD.Approx_V1_1').OMP
OMP_NA_V0 = importlib.import_module('Implementations.AK-SVD.Numba_Approx_V0').OMP
OMP_NA_V1 = importlib.import_module('Implementations.AK-SVD.Numba_Approx_V1').OMP

In [9]:
# -----------------------------
# DATA SETUP
# -----------------------------
D, y = make_regression(
    n_samples=50,
    n_features=300,
    n_targets=20000,
    noise=4,
    random_state=0
)

Y = y.T
D = D.T

n_samples = Y.shape[0]

In [10]:
# -----------------------------
# PARAMS
# -----------------------------
T_0 = 1
N = 10  # timeit runs

functions = [
    ("OMP_serial", OMP_serial),
    # ("OMP_numba_serial", OMP_numba_serial),
    ("OMP_numba_jit", OMP_numba_jit),
    ("OMP_numba_njit", OMP_numba_njit),
    # ("OMP_numba_njit_parallel", OMP_numba_njit_parallel),
    ("OMP_numba_njit_parallel_pbatches", OMP_numba_njit_parallel_pbatches),
    
    ("Approx_V1_1", OMP_A_V1_1),
    ("Numba_Approx_V0", OMP_NA_V0),
    ("Numba_Approx_V1", OMP_NA_V1),   
]

max_threads = numba.get_num_threads()

thread_list = [1]
batch_sizes = [2**i for i in range(1, int(np.log2(n_samples/16)) + 1)]
batch_sizes = [b for b in batch_sizes if b <= n_samples]

print("Threads:", thread_list)
print("Batch sizes:", batch_sizes)

Threads: [1]
Batch sizes: [2, 4, 8, 16, 32, 64, 128, 256, 512, 1024]


In [11]:
# -----------------------------
# WARMUP (compile all functions)
# -----------------------------
for name, fn in functions:
    print(name)
    try:
        fn(Y, T_0, D, batch_size=2, float_dtype=np.float64)
    except Exception as e:
        print(f'{e} \n')
        try:
            fn(Y, T_0, D, batch_size=2)
        except Exception as f:
            print(f'{f} \n')
        

OMP_serial
OMP_serial() got an unexpected keyword argument 'float_dtype' 

OMP_numba_jit
some keyword arguments unexpected 



C:\Users\richa\Jupyter Notebooks\CSE 392 - Parallel Algorithms\kSVD\Experiments\Numba\OMP_Implementations\OMP_numba_parallel.py:349: NumbaPerformanceWarning: '@' is faster on contiguous arrays, called on (Array(float64, 2, 'A', False, aligned=True), Array(float64, 2, 'A', False, aligned=True))
  dot = Dk_new[b] @ Q[b, :, :j]


OMP_numba_njit
some keyword arguments unexpected 

OMP_numba_njit_parallel_pbatches
some keyword arguments unexpected 



C:\Users\richa\Jupyter Notebooks\CSE 392 - Parallel Algorithms\kSVD\Experiments\Numba\OMP_Implementations\OMP_numba_parallel.py:1039: NumbaPerformanceWarning: '@' is faster on contiguous arrays, called on (Array(float64, 2, 'A', False, aligned=True), Array(float64, 2, 'A', False, aligned=True))
  dot = Dk_new[b] @ Q[b, :, :j]
C:\Users\richa\Jupyter Notebooks\CSE 392 - Parallel Algorithms\kSVD\Experiments\Numba\OMP_Implementations\OMP_numba_parallel.py:1039: NumbaPerformanceWarning: '@' is faster on contiguous arrays, called on (Array(float64, 2, 'A', False, aligned=True), Array(float64, 2, 'A', False, aligned=True))
  dot = Dk_new[b] @ Q[b, :, :j]


Approx_V1_1
Numba_Approx_V0
Numba_Approx_V1


In [12]:
# -----------------------------
# BENCHMARK
# -----------------------------
results = []

for name, fn in functions:
    print(f"\n--- {name} ---")
    
    for n_threads in thread_list:
        set_num_threads(n_threads)
        
        for batch_size in batch_sizes:
            # define callable for timeit
            stmt = lambda: fn(Y, T_0, D, batch_size=batch_size)
            stmt2 = lambda: fn(Y, T_0, D, batch_size=batch_size, float_dtype=np.float64)
            
            try:
                stmt2()
            except:
                try:
                    stmt()
                except:
                    print(f"FAIL: {name}, threads={n_threads}, batch={batch_size}")
                    continue
            try:
                t = timeit.timeit(stmt2, number=N) / N
            except Exception as e:
                try:
                    t = timeit.timeit(stmt, number=N) / N               
                except:
                    print(f"FAIL: {name}, threads={n_threads}, batch={batch_size}")
                    continue

            print(f"{name} | threads={n_threads} | batch={batch_size} | {t:.6f}s")

            results.append((name, n_threads, batch_size, t))


--- OMP_serial ---
OMP_serial | threads=1 | batch=2 | 0.813189s
OMP_serial | threads=1 | batch=4 | 0.412316s
OMP_serial | threads=1 | batch=8 | 0.213118s
OMP_serial | threads=1 | batch=16 | 0.109612s
OMP_serial | threads=1 | batch=32 | 0.068902s
OMP_serial | threads=1 | batch=64 | 0.049626s
OMP_serial | threads=1 | batch=128 | 0.038760s
OMP_serial | threads=1 | batch=256 | 0.031247s
OMP_serial | threads=1 | batch=512 | 0.054227s
OMP_serial | threads=1 | batch=1024 | 0.057613s

--- OMP_numba_jit ---
OMP_numba_jit | threads=1 | batch=2 | 0.167827s
OMP_numba_jit | threads=1 | batch=4 | 0.123406s
OMP_numba_jit | threads=1 | batch=8 | 0.108360s
OMP_numba_jit | threads=1 | batch=16 | 0.109728s
OMP_numba_jit | threads=1 | batch=32 | 0.092657s
OMP_numba_jit | threads=1 | batch=64 | 0.091450s
OMP_numba_jit | threads=1 | batch=128 | 0.099568s
OMP_numba_jit | threads=1 | batch=256 | 0.096480s
OMP_numba_jit | threads=1 | batch=512 | 0.144768s
OMP_numba_jit | threads=1 | batch=1024 | 0.139591s

--

In [13]:
# -----------------------------
# SAVE CSV
# -----------------------------
with open("omp_benchmark_all_thread_limit.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["function", "threads", "batch_size", "time_sec"])
    writer.writerows(results)

print("\nSaved to omp_benchmark_all_thread_limit.csv")


Saved to omp_benchmark_all_thread_limit.csv
